# Module B Report: Local API Development, RBAC, and Database Optimisation
 
**Project:** FixIIT Complaint Management System  
**Video Link:** https://www.youtube.com/watch?v=nxTMkarkBH4


## 1. Introduction

This module implements a local database-backed web application for the FixIIT complaint management system. The main problem addressed is how to manage project data securely through a local UI and API while preserving data integrity, enforcing access control, and improving database performance.

The proposed solution uses a Flask-based local API, a web UI for login and portfolio access, and a SQLite database containing both core system tables and project-specific tables. Core data such as users, credentials, and audit logs are stored separately from the complaint-management workflow tables.

The system also applies Role-Based Access Control so that admins, staff, and regular users have different permissions. In addition, all database-changing API operations are logged, and indexing is applied to the most important complaint-listing query to improve performance.


## 2. Local Environment Setup and Data Management

The application runs locally using Flask, SQLite, HTML templates, and a notebook report for documentation. The database is stored in `app/local_database.db`, and the schema is defined through the SQL files in the `sql` folder.

The design separates core system data from project-specific data. Core  tables include `users`, `logs`, and `audit_context`. Project-specific tables include `member`, `complaint`, `assignment`, `maintenance_log`, `feedback`, `notification`, and supporting lookup tables such as `role`, `status`, `priority`, `hostel`, and `location`.

This separation is important because credentials are stored only in the `users` table and are not duplicated in the project tables. The `member` table links to `users` through `user_id`, which preserves data integrity and keeps authentication data separate from business data. All this is to ensure no unnecessary data duplication.


## 3. API and UI Development

The local application provides both a web UI and secured APIs. The UI includes a login page and a portfolio page, while the API supports authenticated operations such as login, complaint retrieval, complaint creation, complaint update, complaint deletion, and portfolio management.

Protected API routes validate the user session using a JWT token. If the token is missing, invalid, or expired, the request is rejected. This ensures that all protected actions are available only to authenticated users. 

The Member Portfolio feature allows authenticated users to view portfolio details. Admin users can view and manage all member records, while regular users are restricted to their own profile and allowed complaint operations.


## 4. Role-Based Access Control and Logging

RBAC is enforced in both the UI behavior and the backend API routes. Admin users can create and delete members, view audit logs, and perform administrative actions. Staff and admin users can update complaint status. Regular users can view their own portfolio and create or manage only their own complaints.

Whenever a database-changing action passes through the API, an audit entry is recorded. Logging is written both to `logs/audit.log` and to the `logs` table in the database. Each entry stores the actor, source, table name, action, and details of the change.

The system also makes unauthorized behavior visible. If a user attempts an action outside their role, the request is rejected and a log entry is created. Direct database modifications can also be identified because the audit context marks non-API changes with `DIRECT_DB`, making them distinguishable during review.


## 5. SQL Indexing and Query Optimisation

The main optimization target is the complaint-listing query used by the `/complaints` API. This query joins `complaint`, `member`, `users`, and `status`, and orders the results using `complaint.created_at DES C`. 

To improve this workflow, indexes were created on the complaint table:

- `idx_complaint_member_id` on `complaint(member_id)`
- `idx_complaint_status_id` on `complaint(status_id)`
- `idx_complaint_created_at` on `complaint(created_at DESC)`

These indexes directly support the join and ordering patterns used in the benchmarked SQL query. The goal is to reduce unnecessary scanning and improve the speed of both SQL execution and API response time. We expect a better performance after this indexing


## 6. Performance Benchmarking

Performance was measured before and after applying indexes. Two metrics were collected: raw SQL execution time for repeated runs of the complaint-listing query, and API response time for repeated authenticated requests to `/complaints`.

| Metric | Before Indexing | After Indexing | Improvement |
| --- | ---: | ---: | ---: |
| SQL execution time (2000 runs) | 0.3161 s | 0.1891 s | 1.67x faster |
| API response time (200 runs) | 0.3988 s | 0.3465 s | 1.15x faster |

The `EXPLAIN QUERY PLAN` output also changed after indexing. Before indexing, SQLite performed a scan on `complaint` and used a temporary B-tree for ordering. After indexing, it used `idx_complaint_created_at`, which removed the temporary B-tree step and improved the query path.


## 7. Benchmark Visualisation

The graph below summarizes the before-and-after benchmark results.

![Benchmark Results](benchmark_results.png)


## 8. Audit Log Evidence

The audit log demonstrates that valid API-based modifications and unauthorized attempts are both recorded. This supports the assignment requirement that all database-changing API calls be logged locally.

Example evidence from the log includes admin member creation, regular-user complaint creation, and unauthorized attempts such as restricted delete actions. Because the logging system stores the actor and source of each modification, it is possible to distinguish normal API activity from direct database changes.


## 9. Conclusion

This module successfully implements a local authenticated system with a web UI, protected APIs, RBAC, audit logging, and indexed database access. The system separates core identity data from project-specific data, which helps maintain integrity and avoids duplication.

The benchmarking results show that indexing improves the complaint-listing workflow at both the SQL and API levels. The audit trail also demonstrates that legitimate writes and unauthorized actions are both visible during review. Overall, the implementation satisfies the required goals of local API development, access control, logging, and database optimisation for the assignment.
